In [1]:
# =============================================================================
# QFID · Q1 + RETRIEVAL-DRIVEN (EPISODIC) PRE-ADAPTATION: Q0 / Q1 / Q2
# Cell 1/4 — config, environment, cohorts
# -----------------------------------------------------------------------------
# Q0/Q1/Q2 now act DURING pre-adaptation, not only at test time.
#   For each metric m in PREADAPT_METRICS, fresh copies of T0_NIH_whole_vit.pt and
#   T0_NIH_patch_vit.pt are fine-tuned JOINTLY on the 1.6k labelled frontal CheXpert cohort:
#     m = "none"        (Q0) : BCE on the Stage-A routed logit mix (no retrieval)
#     m = "cosine"      (Q1) : episodic retrieval loss, cosine similarity
#     m = "euclidean"   (Q2) : episodic retrieval loss, -||q - s||^2 (raw routed vector)
#     m = "mahalanobis" (Q2) : episodic retrieval loss, -(q-s)^T Sigma^-1 (q-s)
#   Episodic loss: inside each labelled batch, every image is a query scored against the
#   other images (leave-one-out) as supports: mean sim to positives - mean sim to negatives,
#   per class, on the ROUTED feature (whole <-> patch, weights from the checkpoints'
#   per_class_validation_metrics.csv) -> BCE with the label. Same formula as test time.
#   CheXzero is pre-adapted once with BCE and only supplies the zero-shot prior.
# Cell 2  metric-driven pre-adaptation (+ CheXzero), feature caches per encoder set
# Cell 3  routing table + frozen audit
# Cell 4  K-shot retrieval Q0/Q1/Q2 on every encoder set (matched metric = headline)
# Leakage: the 1.6k pre-adaptation patients are never few-shot supports ('support' cohort
#   is disjoint); select/dev/final/expert_valid disjoint from both (asserted).
# No quantum element in Q0/Q1/Q2: they are the classical ablations. The fidelity metric
#   slot (PREADAPT_METRICS / METHOD_FNS) is left for the QFid fidelity score.
# Run RUN_MODE="SMOKE" first (~10 min), then "FULL". Results need supervisor review.
# =============================================================================
from __future__ import annotations
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, math, time, random, hashlib, re, subprocess, copy, warnings
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from torchvision.models import vit_b_16
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import t as student_t

try:
    import open_clip
except ImportError:   # needs Kaggle internet ON
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    import open_clip
from open_clip.model import build_model_from_openai_state_dict

# ----------------------------------------------------------------------------
# 1. Paths
# ----------------------------------------------------------------------------
STAGE1_ROOT   = Path("/kaggle/input/datasets/zarinn/nih-checkpoint-vit/qfid_matched_stage1_nih_only")
T0_WHOLE_PATH = STAGE1_ROOT / "T0_NIH_whole_vit.pt"
T0_PATCH_PATH = STAGE1_ROOT / "T0_NIH_patch_vit.pt"
ROUTING_CSV   = STAGE1_ROOT / "per_class_validation_metrics.csv"
CHEXZERO_CKPT = Path("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/best_64_5e-05_original_22000_0.864.pt")
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
OUT = Path("/kaggle/working/qfid_episodic_preadapt"); OUT.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 2. Settings
# ----------------------------------------------------------------------------
RUN_MODE = "FULL"                      # "SMOKE" -> "FULL"
SMOKE    = RUN_MODE == "SMOKE"
SEED     = 42
CLASSES    = ("Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion")
LABEL_COLS = tuple(f"label_{i}" for i in range(len(CLASSES)))
LABEL_POLICY = "mask"                   # Q1 policy: CheXpert -1 and blank ignored

# Cohorts (patients). 'preadapt' == Q1's 'train' cohort (same function, seed, cap).
CAP = dict(preadapt=40, support=40, select=40, dev=40, final=40) if SMOKE else \
      dict(preadapt=1600, support=1600, select=400, dev=400, final=400)
SPLIT_FRACTIONS  = {"train": .55, "select": .15, "dev": .15, "final": .15}
USE_EXPERT_VALID = not SMOKE        # SMOKE skips the 234-image expert set to stay under ~10 min

# Pre-adaptation (labels of the 1.6k cohort)
PREADAPT_METRICS = ("none", "cosine", "euclidean", "mahalanobis")   # Q0, Q1, Q2e, Q2m objectives
ADAPT_EPOCHS   = 1 if SMOKE else 5
ADAPT_BLOCKS   = 2                      # last N transformer blocks + final LN + head are trainable
LR_BACKBONE, LR_HEAD, LR_SCALE = 1e-5, 1e-4, 1e-2   # ViT / CheXzero bias / retrieval temperature+bias
LAMBDA_ANCHOR  = 0.1                    # 1 - cos(student emb, original emb): limits forgetting
BATCH_JOINT, BATCH_CXZ = (10, 8) if SMOKE else (20, 64)  # joint whole+patch batch (20 images = 20 + 120 crops)
PER_CLASS_QUOTA = 1 if SMOKE else 2    # class-balanced batches: >= quota labelled pos AND neg per class per batch
                                        # (quota 2 -> every positive query still has another positive to compare to)
MAHAL_SHRINK, COV_MOMENTUM = 0.1, 0.9
SEL_REF_N      = 20 if SMOKE else 400   # labelled pre-adapt patients used as supports for SELECT model selection
SMOKE_MAX_STEPS = 2
SMOKE_BUDGET_MIN = 10
ENCODER_SETS   = ("original",) + tuple(f"pre_{m}" for m in PREADAPT_METRICS)
MATCHED_METHOD = {"pre_none": "Q0_stageA", "pre_cosine": "Q1_cosine", "pre_euclidean": "Q2_euclidean", "pre_mahalanobis": "Q2_mahalanobis"}

# Routing
ROUTING_SOURCE   = "nih_val_csv"        # "nih_val_csv" (checkpoint validation AUROC) | "select" (Q1 rule, ORIGINAL encoders on SELECT) | "equal"
ROUTING_DEADBAND = 0.03
TARGET_CROP_BOXES = [(.05,.05,.55,.95), (.45,.05,.95,.95), (.10,.02,.90,.58), (.10,.42,.90,.98), (.18,.18,.82,.82), (.28,.08,.72,.92)]

# Runtime
EVAL_BATCH, WORKERS = 16, 0             # workers=0 avoids Kaggle loader deadlocks (Q1)
MAX_RUNTIME_HOURS = 11.0

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU (Settings -> Accelerator -> GPU T4), then Run All.")
DEVICE = torch.device("cuda:0"); torch.cuda.set_device(DEVICE)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
RUN_T0 = time.time()
def elapsed(): return f"t+{(time.time()-RUN_T0)/60:.1f} min"
print("GPU:", torch.cuda.get_device_name(0), "| RUN_MODE:", RUN_MODE)
for p in (T0_WHOLE_PATH, T0_PATCH_PATH, ROUTING_CSV, CHEXZERO_CKPT):
    if not p.is_file(): raise FileNotFoundError(f"missing input: {p}")

def sha_of(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()
def file_sha256(path, block=8 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block): h.update(chunk)
    return h.hexdigest()

# ----------------------------------------------------------------------------
# 3. Metrics / losses
# ----------------------------------------------------------------------------
def macro_metrics(y, s):
    y, s = np.asarray(y, float), np.asarray(s, float); per = {}
    for c, name in enumerate(CLASSES):
        m = np.isfinite(y[:, c]); ok = m.sum() and len(np.unique(y[m, c])) == 2
        per[name] = (roc_auc_score(y[m, c], s[m, c]), average_precision_score(y[m, c], s[m, c])) if ok else (np.nan, np.nan)
    au = [a for a, _ in per.values() if np.isfinite(a)]; ap = [p for _, p in per.values() if np.isfinite(p)]
    return {"macro_auroc": float(np.mean(au)) if au else np.nan, "macro_auprc": float(np.mean(ap)) if ap else np.nan, "per_class": per}
def macro_auc(y, s): return macro_metrics(y, s)["macro_auroc"]

def masked_bce(logits, y):
    known = torch.isfinite(y)
    raw = F.binary_cross_entropy_with_logits(logits.float(), torch.nan_to_num(y, nan=0.0), reduction="none")
    return (raw * known).sum() / known.sum().clamp_min(1)

# ----------------------------------------------------------------------------
# 4. Cohort helpers (verbatim Q1 logic)
# ----------------------------------------------------------------------------
def patient_from_path(p):
    m = re.search(r"(patient\d+)", str(p), re.I)
    return m.group(1).lower() if m else Path(str(p)).parts[-3].lower()

def partition_patients(df, fractions, seed):
    rng = np.random.default_rng(seed)
    Y = df[list(LABEL_COLS)].to_numpy(float)
    chars = np.where(~np.isfinite(Y), "u", np.where(Y >= .5, "1", "0"))
    patterns = pd.Series(["".join(row) for row in chars], index=df.index)
    groups = {k: v.to_numpy() for k, v in patterns.groupby(patterns).groups.items()}
    out = {k: [] for k in fractions}; target = {k: fractions[k] * len(df) for k in fractions}
    for _, idx in sorted(groups.items()):
        for i in rng.permutation(idx):
            key = min(fractions, key=lambda k: (len(out[k]) + 1) / (target[k] + 1e-9)); out[key].append(i)
    frames = {k: df.loc[v].reset_index(drop=True) for k, v in out.items()}
    ids = {k: set(v.patient_id.astype(str)) for k, v in frames.items()}
    for a in frames:
        for b in frames:
            if a < b and ids[a] & ids[b]: raise RuntimeError(f"patient leakage between {a} and {b}")
    return frames

def balanced_cap(df, cap, seed):
    if cap <= 0 or len(df) <= cap: return df.reset_index(drop=True)
    rng = np.random.default_rng(seed); chosen = []
    Y = df[list(LABEL_COLS)].to_numpy(float); per = max(2, cap // (2 * len(CLASSES)))
    for c in range(len(CLASSES)):
        for mask in (np.isfinite(Y[:, c]) & (Y[:, c] >= .5), np.isfinite(Y[:, c]) & (Y[:, c] < .5)):
            idx = np.flatnonzero(mask)
            if len(idx): chosen += rng.choice(idx, min(per, len(idx)), replace=False).tolist()
    chosen = list(dict.fromkeys(chosen))
    rest = np.setdiff1d(np.arange(len(df)), np.asarray(chosen, dtype=int))
    if len(chosen) < cap: chosen += rng.choice(rest, min(cap - len(chosen), len(rest)), replace=False).tolist()
    return df.iloc[sorted(chosen[:cap])].reset_index(drop=True)

def split_seed(name): return SEED + int(hashlib.sha256(name.encode()).hexdigest()[:6], 16) % 1000

def find_chexpert_csv(name):
    """Direct locations first; the recursive search (slow on Kaggle's network mount) only as a fallback."""
    for c in (CHEXPERT_ROOT / name, CHEXPERT_ROOT / "CheXpert-v1.0-small" / name, CHEXPERT_ROOT / "CheXpert-v1.0" / name):
        if c.is_file(): return c
    hits = sorted(CHEXPERT_ROOT.rglob(name)) if CHEXPERT_ROOT.is_dir() else []
    return hits[0] if hits else None

def resolve_cxp(p):
    q = Path(str(p)); cands = [q, CHEXPERT_ROOT / q]
    if len(q.parts) > 1: cands.append(CHEXPERT_ROOT / Path(*q.parts[1:]))
    cands.append(CHEXPERT_ROOT / "CheXpert-v1.0-small" / q)
    for x in cands:
        if x.is_file(): return str(x)
    return None

def load_chexpert(csv_path):
    df = pd.read_csv(csv_path)
    if "Frontal/Lateral" in df: df = df[df["Frontal/Lateral"].astype(str).str.lower().eq("frontal")].copy()   # frontal only
    labels = df[list(CLASSES)].apply(pd.to_numeric, errors="coerce").astype(float).mask(lambda x: x.eq(-1), np.nan)
    for j, col in enumerate(LABEL_COLS): df[col] = labels.iloc[:, j]
    df["patient_id"] = df["Path"].map(patient_from_path)
    df["_pick"] = [hashlib.sha256(f"{SEED}|{p}|{q}".encode()).hexdigest() for p, q in zip(df.patient_id, df.Path)]
    df = df.sort_values(["patient_id", "_pick"]).drop_duplicates("patient_id").drop(columns="_pick")
    return df[np.isfinite(df[list(LABEL_COLS)].to_numpy(float)).any(1)].reset_index(drop=True)

def attach_paths(df, name):
    df = df.copy(); df["resolved_path"] = [resolve_cxp(x) for x in df.Path]
    missing = int(df.resolved_path.isna().sum())
    if missing: raise FileNotFoundError(f"{name}: {missing} CheXpert images not found -- check CHEXPERT_ROOT.")
    return df.reset_index(drop=True)

# ----------------------------------------------------------------------------
# 5. Cohorts
# ----------------------------------------------------------------------------
train_csv = find_chexpert_csv("train.csv")
if train_csv is None: raise FileNotFoundError("CheXpert train.csv not found under CHEXPERT_ROOT.")
FULL_SPLIT = partition_patients(load_chexpert(train_csv), SPLIT_FRACTIONS, SEED)
COHORTS = {}
COHORTS["preadapt"] = attach_paths(balanced_cap(FULL_SPLIT["train"], CAP["preadapt"], split_seed("train")), "preadapt")  # == Q1 'train'
_pre_ids = set(COHORTS["preadapt"].patient_id.astype(str))
_rest = FULL_SPLIT["train"][~FULL_SPLIT["train"].patient_id.astype(str).isin(_pre_ids)].reset_index(drop=True)
COHORTS["support"] = attach_paths(balanced_cap(_rest, CAP["support"], split_seed("support")), "support")
for n in ("select", "dev"):
    COHORTS[n] = attach_paths(balanced_cap(FULL_SPLIT[n], CAP[n], split_seed(n)), n)
valid_csv = find_chexpert_csv("valid.csv") if USE_EXPERT_VALID else None
if valid_csv is not None: COHORTS["expert_valid"] = attach_paths(load_chexpert(valid_csv), "expert_valid")
elif USE_EXPERT_VALID: print("NOTE: valid.csv not found -- expert-label evaluation skipped.")

ids = {k: set(v.patient_id.astype(str)) for k, v in COHORTS.items()}
ids["final_locked"] = set(FULL_SPLIT["final"].patient_id.astype(str))
for a, b in product(ids, ids):
    if a < b and ids[a] & ids[b]: raise RuntimeError(f"patient overlap between {a} and {b}")
print("Cohorts (patients):", {k: len(v) for k, v in COHORTS.items()}, "| FINAL locked, never loaded")
YS = {n: COHORTS[n][list(LABEL_COLS)].to_numpy(float) for n in COHORTS}
cnt = [{"split": n, **{c: f"{int((YS[n][:, i] == 1).sum())}+/{int((YS[n][:, i] == 0).sum())}-" for i, c in enumerate(CLASSES)}} for n in COHORTS]
print("Known labels (pos/neg):\n" + pd.DataFrame(cnt).to_string(index=False))
EVAL_SPLITS = ["support", "select", "dev"] + (["expert_valid"] if "expert_valid" in COHORTS else [])
print("Cell 1 ready.", elapsed())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
GPU: Tesla T4 | RUN_MODE: FULL
Cohorts (patients): {'preadapt': 1600, 'support': 1600, 'select': 400, 'dev': 400, 'expert_valid': 200} | FINAL locked, never loaded
Known labels (pos/neg):
       split Atelectasis Cardiomegaly Consolidation     Edema Pleural Effusion
    preadapt   336+/163-    313+/242-     203+/521- 369+/336-        475+/571-
     support    341+/71-    307+/247-     211+/464- 378+/351-        492+/552-
      select     79+/40-      80+/66-      48+/124-   90+/78-        114+/135-
         dev     87+/40-      71+/73-      54+/138-   94+/79-        127+/146-
expert_valid    75+/125-     66+/134-      32+/168-  42+/158-         64+/136-
Cell 1 ready. t+0.7 min


In [2]:
# =============================================================================
# Cell 2/4 — METRIC-DRIVEN PRE-ADAPTATION (Q0 / Q1 / Q2) on the 1.6k labelled frontal cohort
#   For every metric: fresh T0 whole + T0 patch students, trained JOINTLY (the routed feature
#   needs both), last ADAPT_BLOCKS blocks + final LN + DisentangleHead trainable, anchored to
#   the original embeddings, best epoch by SELECT macro AUROC scored the same way the metric
#   is used later (Q0: Stage-A logits; Q1/Q2: retrieval against SEL_REF_N labelled
#   pre-adapt patients). Original weights kept if no epoch beats them.
#   CheXzero: pre-adapted once with BCE (zero-shot prior only).
# =============================================================================
# ---- T0 ViT branches (Q1 definitions) ----
class DisentangleHead(nn.Module):
    def __init__(self, in_dim=768, shared=256, out=128, n_classes=5):
        super().__init__()
        self.shared   = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, shared), nn.GELU(), nn.Dropout(.1), nn.LayerNorm(shared))
        self.causal   = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.spurious = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.path_head = nn.Linear(out, n_classes)
    def forward(self, h):
        s = self.shared(h); zc = F.normalize(self.causal(s), dim=-1)
        return {"zc": zc, "logits": self.path_head(zc)}

class ViTBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = vit_b_16(weights=None); self.encoder.heads = nn.Identity(); self.head = DisentangleHead()
    def forward(self, x): return self.head(self.encoder(x))

def load_branch(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
    state = {k.removeprefix("module.").removeprefix("model."): v for k, v in state.items() if torch.is_tensor(v)}
    model = ViTBranch(); model.load_state_dict(state, strict=True)
    for p in model.parameters(): p.requires_grad_(False)
    return model.to(DEVICE).eval()

IMNET = ((.485, .456, .406), (.229, .224, .225))
EVAL_T  = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(*IMNET)])
TRAIN_T = T.Compose([T.RandomResizedCrop(224, scale=(.8, 1.), ratio=(.9, 1.1)), T.ToTensor(), T.Normalize(*IMNET)])
PRE_AUG = T.Compose([T.RandomRotation(5), T.ColorJitter(brightness=.2, contrast=.2)])   # no flips (laterality)

class T0DS(Dataset):
    """mode 'whole' -> (3,224,224); mode 'patch' -> (6,3,224,224) TARGET_CROP_BOXES. Labels may contain NaN."""
    def __init__(self, df, mode, train=False):
        self.paths, self.y, self.mode, self.train = df.resolved_path.tolist(), df[list(LABEL_COLS)].to_numpy(np.float32), mode, train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as z: im = z.convert("RGB")
        if self.train: im = PRE_AUG(im)
        if self.mode == "whole": x = (TRAIN_T if self.train else EVAL_T)(im)
        else:
            w, h = im.size
            x = torch.stack([EVAL_T(im.crop((int(a*w), int(b*h), int(c*w), int(d*h)))) for a, b, c, d in TARGET_CROP_BOXES])
        return x, torch.from_numpy(self.y[i])

def lme(logits):   # (N,K,5) -> (N,5) log-mean-exp over crops
    return torch.logsumexp(logits.float(), dim=1) - math.log(logits.shape[1])

@torch.inference_mode()
def encode_t0(model, df, mode, tag=""):
    model.eval(); zc, lg = [], []
    for bi, (x, _) in enumerate(DataLoader(T0DS(df, mode), batch_size=EVAL_BATCH, shuffle=False, num_workers=WORKERS), 1):
        shape = x.shape[:2] if mode == "patch" else x.shape[:1]
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x.flatten(0, 1).to(DEVICE) if mode == "patch" else x.to(DEVICE))
        zc.append(out["zc"].float().view(*shape, -1).cpu()); lg.append(out["logits"].float().view(*shape, -1).cpu())
    return {"zc": torch.cat(zc), "logits": torch.cat(lg)}

def t0_scores(e): return lme(e["logits"]) if e["logits"].dim() == 3 else e["logits"]   # one score per class per image

# ---- CheXzero (CLIP ViT-B/32) ----
def load_chexzero(path):
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and isinstance(sd.get("state_dict"), dict): sd = sd["state_dict"]
    if isinstance(sd, dict) and isinstance(sd.get("model"), dict): sd = sd["model"]
    if not isinstance(sd, dict): sd = sd.state_dict()
    sd = {k.removeprefix("module."): v for k, v in sd.items() if torch.is_tensor(v)}
    model = build_model_from_openai_state_dict(sd).float().to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model

# CheXzero preprocessing (as far as I know from its repo): grayscale, aspect-preserving pad to 320,
# 0-255 scale, normalise (101.48761, 83.43944), bicubic 224, 3 channels. SELECT zero-shot sanity below.
CXZ_MEAN, CXZ_STD = 101.48761, 83.43944
BICUBIC = T.InterpolationMode.BICUBIC
def pad_to_square(path, size=320):
    with Image.open(path) as z: im = z.convert("L")
    w, h = im.size; s = size / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
    canvas = Image.new("L", (size, size), 0); canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
    return canvas
CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
CXZ_MODE = "chexzero"   # auto-chosen below: "chexzero" (0-255, 101.49/83.44) vs "clip" (standard CLIP normalisation)
def cxz_input(im):
    if CXZ_MODE == "chexzero": return ((TF.pil_to_tensor(im).float() - CXZ_MEAN) / CXZ_STD).expand(3, -1, -1).contiguous()
    return TF.normalize(TF.to_tensor(im.convert("RGB")), CLIP_MEAN, CLIP_STD)
CXZ_EVAL = T.Resize((224, 224), interpolation=BICUBIC)
CXZ_AUG  = T.Compose([T.RandomResizedCrop(224, scale=(.8, 1.), ratio=(.9, 1.1), interpolation=BICUBIC),
                      T.RandomRotation(5), T.ColorJitter(brightness=.2, contrast=.2)])

class CXZDS(Dataset):
    def __init__(self, df, train=False):
        self.paths, self.y, self.train = df.resolved_path.tolist(), df[list(LABEL_COLS)].to_numpy(np.float32), train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = pad_to_square(self.paths[i])
        return cxz_input(CXZ_AUG(im) if self.train else CXZ_EVAL(im)), torch.from_numpy(self.y[i])

@torch.inference_mode()
def encode_cxz(visual, df):
    visual.eval(); out = []
    for x, _ in DataLoader(CXZDS(df), batch_size=64, shuffle=False, num_workers=WORKERS):
        with torch.autocast("cuda", dtype=torch.float16): z = visual(x.to(DEVICE))
        out.append(F.normalize(z.float(), dim=-1).cpu())
    return torch.cat(out)

# ---- routing weights (fixed BEFORE pre-adaptation; used inside the retrieval loss) ----
def advantage_weight(a_whole, a_patch, deadband=ROUTING_DEADBAND):
    adv_w = (a_whole - .5) if (a_whole - .5) > deadband else 0.0
    adv_p = (a_patch - .5) if (a_patch - .5) > deadband else 0.0
    return adv_p / (adv_p + adv_w) if (adv_p + adv_w) > 1e-6 else 0.5

def routing_from_csv(path):
    df = pd.read_csv(path)
    df = df[df["tier"].astype(str).eq("T0") & df["stage"].astype(str).str.contains("validation", case=False)]
    piv = df.pivot_table(index="pathology", columns="branch", values="auroc")
    missing = [c for c in CLASSES if c not in piv.index]
    if missing or not {"whole", "patch"} <= set(piv.columns): raise ValueError(f"routing CSV lacks {missing or 'whole/patch columns'}")
    return pd.DataFrame([{"class": c, "auroc_whole": piv.loc[c, "whole"], "auroc_patch": piv.loc[c, "patch"],
                          "patch_weight": advantage_weight(piv.loc[c, "whole"], piv.loc[c, "patch"])} for c in CLASSES])

def routing_from_select(w_model, p_model):
    """Q1 rule on CheXpert SELECT with the ORIGINAL (not yet adapted) encoders."""
    y = YS["select"]; w = encode_t0(w_model, COHORTS["select"], "whole")["logits"].numpy()
    pl = encode_t0(p_model, COHORTS["select"], "patch")["logits"]; p = (torch.softmax(pl, 1) * pl).sum(1).numpy()
    rows = []
    for c, name in enumerate(CLASSES):
        m = np.isfinite(y[:, c]); ok = m.sum() and len(np.unique(y[m, c])) == 2
        aw = roc_auc_score(y[m, c], w[m, c]) if ok else .5; ap = roc_auc_score(y[m, c], p[m, c]) if ok else .5
        rows.append({"class": name, "auroc_whole": aw, "auroc_patch": ap, "patch_weight": advantage_weight(aw, ap)})
    return pd.DataFrame(rows)

def route(whole_zc, patch_zc, patch_logits, normalize):
    """Per-class routed feature (B,5,D): w_p * attention-pooled patch_c + w_w * whole (Q1 routing)."""
    attn = torch.softmax(patch_logits.float(), dim=1)                                   # (B,6,5)
    pf = torch.einsum("bkc,bkd->bcd", attn, patch_zc.float())                          # (B,5,D)
    pw = PATCH_W.to(pf.device)[None, :, None]
    f = pw * pf + (1 - pw) * whole_zc.float()[:, None, :]
    return F.normalize(f, dim=-1) if normalize else f

def stage_a(whole_logits, patch_logits):
    """Q0 score: routed mix of branch logits (no retrieval)."""
    ps = (torch.softmax(patch_logits.float(), 1) * patch_logits.float()).sum(1)
    pw = PATCH_W.to(ps.device)[None]
    return pw * ps + (1 - pw) * whole_logits.float()

# ---- retrieval primitives (shared by the training loss and SELECT scoring) ----
def pair_sim(metric, Q, S, prec=None):
    """Q (n,5,D), S (m,5,D) -> (n,m,5) similarity, higher = closer."""
    if metric == "cosine": return torch.einsum("ncd,mcd->nmc", Q, S)
    d = Q[:, None] - S[None]                                                          # (n,m,5,D)
    if metric == "euclidean": return -(d ** 2).sum(-1)
    return -torch.einsum("nmcd,cde,nmce->nmc", d, prec, d)

def retrieval_score(metric, Q, S, ys, prec=None, loo=False):
    """score (n,5) = mean sim to labelled positives - mean sim to labelled negatives among S.
       loo=True: Q is S (same batch) and each query's own entry is excluded."""
    sim = pair_sim(metric, Q, S, prec)
    known = torch.isfinite(ys)
    pos = (known & (ys >= .5)).float()[None].expand(len(Q), -1, -1)
    neg = (known & (ys < .5)).float()[None].expand(len(Q), -1, -1)
    if loo:
        keep = (~torch.eye(len(Q), dtype=torch.bool, device=Q.device))[:, :, None].float()
        pos, neg = pos * keep, neg * keep
    npos, nneg = pos.sum(1), neg.sum(1)
    score = (sim * pos).sum(1) / npos.clamp_min(1) - (sim * neg).sum(1) / nneg.clamp_min(1)
    return score, (npos > 0) & (nneg > 0)

def batch_cov(X):
    Xc = X - X.mean(0, keepdim=True)
    return torch.einsum("bcd,bce->cde", Xc, Xc) / max(1, len(X) - 1)                  # (5,D,D)

def precision(cov, shrink=MAHAL_SHRINK):
    D = cov.shape[-1]; tr = cov.diagonal(dim1=-2, dim2=-1).mean(-1)[:, None, None]
    return torch.linalg.inv((1 - shrink) * cov + shrink * tr.clamp_min(1e-6) * torch.eye(D, device=cov.device))

@torch.no_grad()
def retrieval_chunked(metric, Q, S, ys, prec=None, chunk=32):
    return torch.cat([retrieval_score(metric, Q[i:i + chunk], S, ys, prec)[0] for i in range(0, len(Q), chunk)])

class JointDS(Dataset):
    """Training view for joint pre-adaptation: augmented whole image + six crops of the SAME augmented image."""
    def __init__(self, df): self.paths, self.y = df.resolved_path.tolist(), df[list(LABEL_COLS)].to_numpy(np.float32)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as z: im = PRE_AUG(z.convert("RGB"))
        w, h = im.size
        crops = torch.stack([EVAL_T(im.crop((int(a*w), int(b*h), int(c*w), int(d*h)))) for a, b, c, d in TARGET_CROP_BOXES])
        return TRAIN_T(im), crops, torch.from_numpy(self.y[i])

class BalancedBatchSampler(torch.utils.data.Sampler):
    """Every batch holds >= PER_CLASS_QUOTA labelled positives AND negatives per class (when the cohort has them),
       no duplicates (duplicates would leak a query into its own support under leave-one-out), rest random."""
    def __init__(self, Y, batch_size, n_batches, seed):
        self.n, self.bs, self.nb, self.rng = len(Y), batch_size, n_batches, np.random.default_rng(seed)
        known = np.isfinite(Y)
        self.pools = [(np.flatnonzero(known[:, c] & (Y[:, c] >= .5)), np.flatnonzero(known[:, c] & (Y[:, c] < .5))) for c in range(Y.shape[1])]
        need = 2 * PER_CLASS_QUOTA * Y.shape[1]
        if batch_size < need: raise ValueError(f"BATCH_JOINT={batch_size} < 2*quota*classes={need}")
    def __len__(self): return self.nb
    def __iter__(self):
        for _ in range(self.nb):
            b, seen = [], set()
            for c in self.rng.permutation(len(self.pools)):
                for pool in self.pools[c]:
                    cand = [i for i in self.rng.permutation(pool) if i not in seen][:PER_CLASS_QUOTA]
                    b += [int(i) for i in cand]; seen.update(cand)
            for i in self.rng.permutation(self.n):
                if len(b) >= self.bs: break
                if i not in seen: b.append(int(i)); seen.add(i)
            yield b[:self.bs]

def set_trainable_vit(model):
    for p in model.parameters(): p.requires_grad_(False)
    params = list(model.encoder.encoder.layers[-ADAPT_BLOCKS:].parameters()) + list(model.encoder.encoder.ln.parameters()) + list(model.head.parameters())
    for p in params: p.requires_grad_(True)
    return params

ORIG = {"whole": load_branch(T0_WHOLE_PATH), "patch": load_branch(T0_PATCH_PATH)}
if ROUTING_SOURCE == "nih_val_csv": ROUTING_DF = routing_from_csv(ROUTING_CSV)
elif ROUTING_SOURCE == "select":    ROUTING_DF = routing_from_select(ORIG["whole"], ORIG["patch"])
elif ROUTING_SOURCE == "equal":     ROUTING_DF = pd.DataFrame({"class": CLASSES, "patch_weight": [.5] * len(CLASSES)})
else: raise ValueError(ROUTING_SOURCE)
PATCH_W = torch.tensor(ROUTING_DF.patch_weight.to_numpy(), dtype=torch.float32)
print(f"=== ROUTING (source={ROUTING_SOURCE}) ===\n" + ROUTING_DF.round(3).to_string(index=False))
CKPT_SHA = {"whole": file_sha256(T0_WHOLE_PATH), "patch": file_sha256(T0_PATCH_PATH), "chexzero": file_sha256(CHEXZERO_CKPT)}
SEL_REF = COHORTS["preadapt"].sample(n=min(SEL_REF_N, len(COHORTS["preadapt"])), random_state=SEED + 3).reset_index(drop=True)
Y_REF = torch.tensor(SEL_REF[list(LABEL_COLS)].to_numpy(np.float32), device=DEVICE)

def encode_pair(w_model, p_model, df):
    return {"whole": encode_t0(w_model, df, "whole"), "patch": encode_t0(p_model, df, "patch")}

def select_auc(metric, w_model, p_model):
    q = encode_pair(w_model, p_model, COHORTS["select"])
    if metric == "none": return macro_auc(YS["select"], stage_a(q["whole"]["logits"], q["patch"]["logits"]).numpy())
    r = encode_pair(w_model, p_model, SEL_REF)
    norm = metric == "cosine"
    Fq = route(q["whole"]["zc"], q["patch"]["zc"], q["patch"]["logits"], norm).to(DEVICE)
    Fs = route(r["whole"]["zc"], r["patch"]["zc"], r["patch"]["logits"], norm).to(DEVICE)
    prec = precision(batch_cov(Fs)) if metric == "mahalanobis" else None
    return macro_auc(YS["select"], retrieval_chunked(metric, Fq, Fs, Y_REF, prec).cpu().numpy())

def preadapt_fingerprint(metric):
    return sha_of({"metric": metric, "ckpt": CKPT_SHA, "routing": PATCH_W.tolist(), "run_mode": RUN_MODE, "seed": SEED,
                   "patients": COHORTS["preadapt"].patient_id.astype(str).tolist(),
                   "cfg": [ADAPT_EPOCHS, ADAPT_BLOCKS, LR_BACKBONE, LR_SCALE, LAMBDA_ANCHOR, BATCH_JOINT, PER_CLASS_QUOTA,
                           MAHAL_SHRINK, COV_MOMENTUM, SEL_REF_N]})[:12]

def preadapt_metric(metric):
    seed_all(SEED)                                   # every metric starts from identical randomness
    w_st, p_st = copy.deepcopy(ORIG["whole"]), copy.deepcopy(ORIG["patch"])
    trainable = set_trainable_vit(w_st) + set_trainable_vit(p_st)
    log_tau = nn.Parameter(torch.zeros((), device=DEVICE)); bias = nn.Parameter(torch.zeros(len(CLASSES), device=DEVICE))
    extra = [log_tau, bias]
    opt = torch.optim.AdamW([{"params": trainable, "lr": LR_BACKBONE}, {"params": extra, "lr": LR_SCALE, "weight_decay": 0.0}], weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    loader = DataLoader(JointDS(COHORTS["preadapt"]), num_workers=WORKERS,
                        batch_sampler=BalancedBatchSampler(YS["preadapt"], BATCH_JOINT, max(1, len(COHORTS["preadapt"]) // BATCH_JOINT), SEED))
    base = select_auc(metric, ORIG["whole"], ORIG["patch"])
    best = {"auc": base, "epoch": 0, "w": {k: v.detach().cpu().clone() for k, v in ORIG["whole"].state_dict().items()},
            "p": {k: v.detach().cpu().clone() for k, v in ORIG["patch"].state_dict().items()}}
    hist = [{"metric": metric, "epoch": 0, "select_auroc": base}]
    cov = COV0.clone() if metric == "mahalanobis" else None      # start from label-free reference covariance, then EMA
    print(f"\n[{metric}] joint pre-adaptation | trainable {sum(p.numel() for p in trainable):,} | SELECT before = {base:.4f} | {elapsed()}")
    for ep in range(1, ADAPT_EPOCHS + 1):
        w_st.train(); p_st.train(); t0 = time.time(); L, A, V = [], [], []
        for step, (xw, xp, y) in enumerate(loader, 1):
            if (time.time() - RUN_T0) / 3600 > MAX_RUNTIME_HOURS:
                raise RuntimeError("MAX_RUNTIME_HOURS reached -- re-run: finished metrics resume from their saved checkpoints.")
            xw, xp, y = xw.to(DEVICE, non_blocking=True), xp.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            B, K = xp.shape[:2]; xpf = xp.flatten(0, 1)
            with torch.autocast("cuda", dtype=torch.float16):
                ow, op = w_st(xw), p_st(xpf)
                with torch.no_grad(): aw, ap = ORIG["whole"](xw)["zc"], ORIG["patch"](xpf)["zc"]
            anchor = 0.5 * (F.cosine_similarity(ow["zc"].float(), aw.float(), dim=-1).mean()
                            + F.cosine_similarity(op["zc"].float(), ap.float(), dim=-1).mean())
            pz, pl = op["zc"].float().view(B, K, -1), op["logits"].float().view(B, K, -1)
            if metric == "none":
                task, frac = masked_bce(stage_a(ow["logits"], pl), y), 1.0
            else:
                Fb = route(ow["zc"], pz, pl, normalize=(metric == "cosine"))
                prec = None
                if metric == "mahalanobis":
                    bc = batch_cov(Fb.detach()); cov = bc if cov is None else COV_MOMENTUM * cov + (1 - COV_MOMENTUM) * bc
                    prec = precision(cov)
                score, valid = retrieval_score(metric, Fb, Fb, y, prec, loo=True)
                use = valid & torch.isfinite(y); frac = use.float().mean().item()
                if use.any():
                    sd = score[use].detach().std() if int(use.sum()) > 1 else torch.ones((), device=DEVICE)
                    z = score / torch.nan_to_num(sd, nan=1.0).clamp_min(1e-6)
                    task = masked_bce(log_tau.exp() * z + bias, y.masked_fill(~use, float("nan")))
                else:
                    task = score.sum() * 0
            loss = task + LAMBDA_ANCHOR * (1 - anchor)
            if not torch.isfinite(loss): raise FloatingPointError(f"[{metric}] non-finite loss at epoch {ep} step {step}")
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + extra, 1.0); scaler.step(opt); scaler.update()
            L.append(loss.item()); A.append(anchor.item()); V.append(frac)
            if ep == 1 and step == 1:
                print(f"  [sanity:{metric}] step 1 loss={loss.item():.4f} task={task.item():.4f} anchor_cos={anchor.item():.4f} "
                      f"usable (query,class) pairs={frac:.2f}")
            if SMOKE and step >= SMOKE_MAX_STEPS: break
            if step % 25 == 0: print(f"  [{metric}] ep{ep} step {step}/{len(loader)} loss={np.mean(L[-25:]):.4f} | {elapsed()}")
        auc = select_auc(metric, w_st, p_st)
        hist.append({"metric": metric, "epoch": ep, "select_auroc": auc, "loss": float(np.mean(L)), "anchor_cos": float(np.mean(A)),
                     "usable_pairs": float(np.mean(V)), "tau": float(log_tau.detach().exp())})
        print(f"[{metric}] epoch {ep}/{ADAPT_EPOCHS} loss={np.mean(L):.4f} anchor={np.mean(A):.4f} usable={np.mean(V):.2f} "
              f"tau={log_tau.exp().item():.2f} SELECT={auc:.4f} | {time.time()-t0:.0f}s | {elapsed()}")
        if metric != "none" and np.mean(V) < 0.5:
            print(f"  WARNING [{metric}]: only {np.mean(V):.2f} of (query, class) pairs had both a positive and a negative support"
                  " in the batch -- raise BATCH_JOINT or PER_CLASS_QUOTA.")
        if np.isfinite(auc) and auc > best["auc"]:
            best = {"auc": auc, "epoch": ep, "w": {k: v.detach().cpu().clone() for k, v in w_st.state_dict().items()},
                    "p": {k: v.detach().cpu().clone() for k, v in p_st.state_dict().items()}}
    del opt, w_st, p_st; torch.cuda.empty_cache()
    print(f"[{metric}] selected epoch {best['epoch']} (SELECT {best['auc']:.4f})" + ("" if best["epoch"] else " -> original weights kept"))
    return best, hist

_r0 = encode_pair(ORIG["whole"], ORIG["patch"], SEL_REF)
COV0 = batch_cov(route(_r0["whole"]["zc"].clone(), _r0["patch"]["zc"].clone(), _r0["patch"]["logits"].clone(), normalize=False).to(DEVICE))
ENC, SELECTED, HISTORY = {}, {}, []
ENC["original"] = {n: encode_pair(ORIG["whole"], ORIG["patch"], COHORTS[n]) for n in EVAL_SPLITS}
ENC["original"] = {b: {n: ENC["original"][n][b] for n in EVAL_SPLITS} for b in ("whole", "patch")}   # ENC[set][branch][split]
for metric in PREADAPT_METRICS:
    fp = preadapt_fingerprint(metric); ckpt_path = OUT / f"T1_CXP_{metric}_whole_patch_{fp}.pt"
    if ckpt_path.is_file():                        # resume: this metric already finished with identical settings
        saved = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        best = {"w": saved["whole"], "p": saved["patch"], "epoch": saved["selected_epoch"]}
        print(f"[{metric}] restored finished pre-adaptation from {ckpt_path.name}")
    else:
        best, hist = preadapt_metric(metric); HISTORY += hist
        torch.save({"whole": best["w"], "patch": best["p"], "metric": metric, "selected_epoch": best["epoch"], "routing": PATCH_W.tolist(),
                    "source_ckpt_sha": CKPT_SHA, "preadapt_patients": COHORTS["preadapt"].patient_id.astype(str).tolist()}, ckpt_path)
    SELECTED[metric] = best["epoch"]
    w_m, p_m = copy.deepcopy(ORIG["whole"]), copy.deepcopy(ORIG["patch"])
    w_m.load_state_dict(best["w"]); p_m.load_state_dict(best["p"])
    enc = {n: encode_pair(w_m.eval(), p_m.eval(), COHORTS[n]) for n in EVAL_SPLITS}
    ENC[f"pre_{metric}"] = {b: {n: enc[n][b] for n in EVAL_SPLITS} for b in ("whole", "patch")}
    del w_m, p_m; torch.cuda.empty_cache()
    print(f"[{metric}] features cached for {EVAL_SPLITS} | {elapsed()}")
pd.DataFrame(HISTORY).to_csv(OUT / "preadapt_history.csv", index=False)

# ---- CheXzero: BCE pre-adaptation once; supplies the zero-shot prior for every encoder set ----
CXZ = {"original": {}, "preadapted": {}}
def cxz_set(enc): return "original" if enc == "original" else "preadapted"

def preadapt_cxz(orig, trainable, extra, loader, step_fn, select_fn):
    student = STUDENTS["chexzero"]
    opt = torch.optim.AdamW([{"params": trainable, "lr": LR_BACKBONE}, {"params": extra, "lr": LR_HEAD}], weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    best = {"auc": select_fn(orig), "epoch": 0, "state": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}}
    for ep in range(1, ADAPT_EPOCHS + 1):
        student.train(); L = []
        for step, (x, y) in enumerate(loader, 1):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                task, s_emb, o_emb = step_fn(student, x, y)
            loss = task + LAMBDA_ANCHOR * (1 - F.cosine_similarity(s_emb.float(), o_emb.float(), dim=-1).mean())
            if not torch.isfinite(loss): raise FloatingPointError(f"[chexzero] non-finite loss at epoch {ep} step {step}")
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + extra, 1.0); scaler.step(opt); scaler.update(); L.append(loss.item())
            if SMOKE and step >= SMOKE_MAX_STEPS: break
        auc = select_fn(student)
        print(f"[chexzero] epoch {ep}/{ADAPT_EPOCHS} loss={np.mean(L):.4f} SELECT zero-shot={auc:.4f} | {elapsed()}")
        if np.isfinite(auc) and auc > best["auc"]:
            best = {"auc": auc, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}}
    del opt; torch.cuda.empty_cache()
    return best["state"], best["epoch"]

STUDENTS = {}
# ---- CheXzero ----
clip_model = load_chexzero(CHEXZERO_CKPT)
with torch.no_grad():   # no_grad (not inference_mode): W_TXT is used inside autograd during pre-adaptation
    names = [c.lower() for c in CLASSES]
    tt = F.normalize(clip_model.encode_text(open_clip.tokenize(names + [f"no {n}" for n in names]).to(DEVICE)).float(), dim=-1)
TPOS, TNEG = tt[:len(CLASSES)].cpu(), tt[len(CLASSES):].cpu()
W_TXT = (TPOS - TNEG).to(DEVICE)                                            # (5,512), frozen
cxz_orig = copy.deepcopy(clip_model.visual).eval(); del clip_model; torch.cuda.empty_cache()
def zs_scores(Z): return (100.0 * Z @ (TPOS - TNEG).T).numpy()              # zero-shot logit(pos) - logit(neg)

STUDENTS["chexzero"] = copy.deepcopy(cxz_orig)
for p in STUDENTS["chexzero"].parameters(): p.requires_grad_(False)
cxz_train = list(STUDENTS["chexzero"].transformer.resblocks[-ADAPT_BLOCKS:].parameters()) + list(STUDENTS["chexzero"].ln_post.parameters())
for p in cxz_train: p.requires_grad_(True)
cxz_bias = nn.Parameter(torch.zeros(len(CLASSES), device=DEVICE))          # calibration only; AUROC-invariant
def cxz_step(student, x, y):
    s = student(x)
    with torch.no_grad(): o = cxz_orig(x)
    return masked_bce(100.0 * F.normalize(s.float(), dim=-1) @ W_TXT.T + cxz_bias, y), s, o
cxz_select = lambda m: macro_auc(YS["select"], zs_scores(encode_cxz(m, COHORTS["select"])))
zs_by_mode = {}
for CXZ_MODE in ("chexzero", "clip"): zs_by_mode[CXZ_MODE] = cxz_select(cxz_orig)
CXZ_MODE = max(zs_by_mode, key=zs_by_mode.get); zs_sel0 = zs_by_mode[CXZ_MODE]
print(f"[chexzero] zero-shot SELECT AUROC by preprocessing: { {k: round(v, 4) for k, v in zs_by_mode.items()} } -> using '{CXZ_MODE}'")
if zs_sel0 < 0.70: print("WARNING: low CheXzero zero-shot with both preprocessings -- check the checkpoint before trusting the prior.")
cxz_loader = DataLoader(CXZDS(COHORTS["preadapt"], train=True), batch_size=BATCH_CXZ, shuffle=True, num_workers=WORKERS,
                        drop_last=True, generator=torch.Generator().manual_seed(SEED))
state, SELECTED["chexzero"] = preadapt_cxz(cxz_orig, cxz_train, [cxz_bias], cxz_loader, cxz_step, cxz_select)
cxz_adapted = copy.deepcopy(cxz_orig); cxz_adapted.load_state_dict(state); cxz_adapted.eval()
torch.save({"visual_state": state, "selected_epoch": SELECTED["chexzero"], "source_ckpt_sha": CKPT_SHA["chexzero"]}, OUT / "T1_CXP_chexzero_visual.pt")
for e, m in (("original", cxz_orig), ("preadapted", cxz_adapted)):
    for n in EVAL_SPLITS: CXZ[e][n] = encode_cxz(m, COHORTS[n])
del cxz_orig, cxz_adapted, STUDENTS["chexzero"]; torch.cuda.empty_cache()
print("Selected pre-adaptation epochs:", SELECTED, "| encoder sets:", list(ENC), "| Cell 2 ready.", elapsed())

=== ROUTING (source=nih_val_csv) ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541

[none] joint pre-adaptation | trainable 28,886,282 | SELECT before = 0.5812 | t+3.4 min
  [sanity:none] step 1 loss=1.1686 task=1.1678 anchor_cos=0.9914 usable (query,class) pairs=1.00
  [none] ep1 step 25/80 loss=0.8012 | t+4.0 min
  [none] ep1 step 50/80 loss=0.6575 | t+4.6 min
  [none] ep1 step 75/80 loss=0.6265 | t+5.3 min
[none] epoch 1/5 loss=0.6861 anchor=0.3880 usable=1.00 tau=1.00 SELECT=0.7962 | 132s | t+5.6 min
  [none] ep2 step 25/80 loss=0.5740 | t+6.2 min
  [none] ep2 step 50/80 loss=0.5685 | t+6.8 min
  [none] ep2 step 75/80 loss=0.5600 | t+7.4 min
[none] epoch 2/5 loss=0.5652 anchor=0.4103 usable=

In [3]:
# =============================================================================
# Cell 3/4 — routing table + frozen audit for every encoder set (no few-shot training yet)
#   original        : NIH T0 checkpoints, no pre-adaptation
#   pre_none   (Q0) : pre-adapted with BCE on Stage-A logits
#   pre_cosine (Q1) / pre_euclidean, pre_mahalanobis (Q2): pre-adapted with the retrieval loss
# =============================================================================
print(f"=== ROUTING used in pre-adaptation and retrieval (source={ROUTING_SOURCE}) ===\n" + ROUTING_DF.round(3).to_string(index=False))
audit = []
for e, n in product(ENCODER_SETS, [s for s in EVAL_SPLITS if s != "support"]):
    y = YS[n]; w, p = ENC[e]["whole"][n], ENC[e]["patch"][n]
    ps = (torch.softmax(p["logits"].clone(), 1) * p["logits"].clone()).sum(1).numpy()
    audit.append({"encoder": e, "split": n, "whole": macro_auc(y, w["logits"].numpy()), "patch": macro_auc(y, ps),
                  "stageA_mix (Q0 score)": macro_auc(y, stage_a(w["logits"].clone(), p["logits"].clone()).numpy()),
                  "chexzero_zs": macro_auc(y, zs_scores(CXZ[cxz_set(e)][n]))})
AUDIT = pd.DataFrame(audit); AUDIT.to_csv(OUT / "frozen_audit.csv", index=False)
print("\n=== FROZEN AUDIT, macro AUROC (branch logits; before any K-shot step) ===\n" + AUDIT.round(4).to_string(index=False))
print("Reference, Q1 NIH-only frozen on DEV: whole 0.589 | patch 0.501 | routed 0.592")
print("Note: for pre_cosine / pre_euclidean / pre_mahalanobis the branch LOGITS are not what was optimised;"
      " their retrieval quality is measured in Cell 4.")
print("Cell 3 ready.", elapsed())

=== ROUTING used in pre-adaptation and retrieval (source=nih_val_csv) ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541

=== FROZEN AUDIT, macro AUROC (branch logits; before any K-shot step) ===
        encoder        split  whole  patch  stageA_mix (Q0 score)  chexzero_zs
       original       select 0.5832 0.4876                 0.5812       0.8940
       original          dev 0.5895 0.5070                 0.5788       0.8772
       original expert_valid 0.5984 0.4314                 0.5591       0.8571
       pre_none       select 0.7764 0.8113                 0.8347       0.9013
       pre_none          dev 0.7476 0.7652                 0.7925       0.8907
       pre_none expert_valid 0.7073

In [4]:
# =============================================================================
# Cell 4/4 — K-shot RETRIEVAL on every encoder set (Q0 / Q1 / Q2), same protocol as before
#   Each pre-adapted encoder set is evaluated with ALL four scoring rules (full grid);
#   the headline is the MATCHED pair (encoder trained with metric m, scored with metric m):
#     pre_none -> Q0_stageA | pre_cosine -> Q1_cosine | pre_euclidean -> Q2_euclidean
#     pre_mahalanobis -> Q2_mahalanobis
#   Supports: K pos + K neg labelled patients per class from the 'support' cohort (Q1 sampler,
#   identical patients per episode for every encoder/method). SELECT-only tuning of beta.
#   Q2_mahalanobis at test time: per-class Ledoit-Wolf covariance of support-cohort features.
#   STRICT by default (RESIDUAL_PRIOR="none"): every Q0/Q1/Q2 number comes ONLY from the pre-adapted
#   T0 whole+patch features -- CheXzero is never blended in. CheXzero zero-shot is logged as a
#   separate reference row (REF_chexzero_zeroshot), never used by Q0/Q1/Q2 or their tests.
#   RESIDUAL_PRIOR="chexzero" re-enables the "+prior" variant (CheXzero logit + beta * score) for context.
#   No quantum element here: these are the classical ablations for the fidelity method.
# =============================================================================
from sklearn.covariance import LedoitWolf

RESIDUAL_PRIOR  = "none"                    # STRICT: Q0/Q1/Q2 use no CheXzero help. "chexzero" -> also report "+prior" blends
Q_SHOTS         = (1,) if SMOKE else (1, 5)
Q_EPISODES      = 2 if SMOKE else 10
N_TUNE_EPISODES = 1 if SMOKE else 5
BETA_GRID       = (0.0, 0.25, 0.5, 1.0, 2.0, 4.0)
PERMUTATIONS    = 100 if SMOKE else 5000
Q_ENCODERS      = ENCODER_SETS
Q_TEST_SPLITS   = [s for s in ("dev", "expert_valid") if s in COHORTS]
RDIR = OUT / "retrieval_episodic"; RDIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 1. Inputs (numpy): normalised + raw routed features, Stage-A logits, prior
# ----------------------------------------------------------------------------
def routed_raw(enc, split):
    p, w = ENC[enc]["patch"][split], ENC[enc]["whole"][split]
    return route(w["zc"].clone(), p["zc"].clone(), p["logits"].clone(), normalize=False).numpy()      # (N,5,128)

def stage_a_logits(enc, split):
    p, w = ENC[enc]["patch"][split], ENC[enc]["whole"][split]
    return stage_a(w["logits"].clone(), p["logits"].clone()).numpy()

RAW   = {e: {s: routed_raw(e, s) for s in EVAL_SPLITS} for e in Q_ENCODERS}
NORM  = {e: {s: RAW[e][s] / np.linalg.norm(RAW[e][s], axis=-1, keepdims=True) for s in EVAL_SPLITS} for e in Q_ENCODERS}
STAGEA = {e: {s: stage_a_logits(e, s) for s in EVAL_SPLITS} for e in Q_ENCODERS}
PRIOR = {e: {s: zs_scores(CXZ[cxz_set(e)][s]) if RESIDUAL_PRIOR == "chexzero" else np.zeros((len(YS[s]), len(CLASSES)))
             for s in EVAL_SPLITS} for e in Q_ENCODERS}
PREC  = {e: [LedoitWolf().fit(RAW[e]["support"][:, c]).precision_ for c in range(len(CLASSES))] for e in Q_ENCODERS}
Y_POOL, Y_SEL = YS["support"], YS["select"]

# ----------------------------------------------------------------------------
# 2. Supports: identical RNG sequence to Q1's sample_support -> same patients as the quantum run
# ----------------------------------------------------------------------------
def draw_support(Y, k, rng):
    sup = {}
    for c in range(len(CLASSES)):
        known = np.isfinite(Y[:, c])
        pos, neg = np.flatnonzero(known & (Y[:, c] >= .5)), np.flatnonzero(known & (Y[:, c] < .5))
        if min(len(pos), len(neg)) < k: raise RuntimeError(f"{CLASSES[c]}: pool too small for {k}-shot -- raise CAP")
        sup[c] = (rng.choice(pos, k, replace=False), rng.choice(neg, k, replace=False))
    return sup

def support_rng(k, episode, tuning=False):
    return np.random.default_rng(SEED + 7919 * int(k) + 101 * episode + (500_000 if tuning else 0))

# ----------------------------------------------------------------------------
# 3. Retrieval scores
# ----------------------------------------------------------------------------
def _sim(metric, q, A, M=None):
    """q: (Nq,D), A: (K,D) -> (Nq,K) similarity (higher = closer)."""
    if metric == "cosine":    return q @ A.T
    d = q[:, None, :] - A[None, :, :]
    if metric == "euclidean": return -(d ** 2).sum(-1)
    return -np.einsum("nkd,de,nke->nk", d, M, d)                                           # mahalanobis

def retrieval(metric, enc, split, sup):
    feats = NORM if metric == "cosine" else RAW
    Fq, Fs = feats[enc][split], feats[enc]["support"]
    S = np.zeros((len(Fq), len(CLASSES)))
    for c, (pi, ni) in sup.items():
        M = PREC[enc][c] if metric == "mahalanobis" else None
        S[:, c] = _sim(metric, Fq[:, c], Fs[pi, c], M).mean(1) - _sim(metric, Fq[:, c], Fs[ni, c], M).mean(1)
    return S

METHOD_FNS = {   # name -> fn(enc, split, sup) -> (N,5) raw score
    "Q0_stageA":      lambda e, s, sup: STAGEA[e][s],
    "Q1_cosine":      lambda e, s, sup: retrieval("cosine", e, s, sup),
    "Q2_euclidean":   lambda e, s, sup: retrieval("euclidean", e, s, sup),
    "Q2_mahalanobis": lambda e, s, sup: retrieval("mahalanobis", e, s, sup),
}

def scaled(name, enc, split, sup):
    """Score divided by its per-class std over the support cohort (label-free) -> common scale."""
    ref = METHOD_FNS[name](enc, "support", sup).std(0); ref[ref < 1e-12] = 1.0
    return METHOD_FNS[name](enc, split, sup) / ref

# ----------------------------------------------------------------------------
# 4. beta for the +prior variant, tuned on SELECT (supports from tuning episodes)
# ----------------------------------------------------------------------------
BETA = {}
for enc, k, name in product(Q_ENCODERS, Q_SHOTS, METHOD_FNS):
    if RESIDUAL_PRIOR == "none": BETA[(enc, k, name)] = 1.0; continue
    tune = [draw_support(Y_POOL, k, support_rng(k, te, tuning=True)) for te in range(N_TUNE_EPISODES)]
    zs = [scaled(name, enc, "select", s) for s in tune]
    auc = {b: np.mean([macro_auc(Y_SEL, PRIOR[enc]["select"] + b * z) for z in zs]) for b in BETA_GRID}
    BETA[(enc, k, name)] = max(auc, key=lambda b: (auc[b], -b))                          # ties -> smaller beta
print("beta (+prior variant, tuned on SELECT):", {f"{e}|{k}|{n}": b for (e, k, n), b in BETA.items()})

# ----------------------------------------------------------------------------
# 5. Sweep: shot x episode x encoder x method (support shared -> paired)
# ----------------------------------------------------------------------------
R_ROWS = []
for k in Q_SHOTS:
    for ep in range(Q_EPISODES):
        sup = draw_support(Y_POOL, k, support_rng(k, ep))
        for s in Q_TEST_SPLITS:                                                   # reference only, never blended
            for cz in ("original", "preadapted"):
                m = macro_metrics(YS[s], zs_scores(CXZ[cz][s]))
                R_ROWS.append({"encoder": f"chexzero_{cz}", "shot": k, "episode": ep, "head": "REF_chexzero_zeroshot", "split": s,
                               "macro_auroc": m["macro_auroc"], "macro_auprc": m["macro_auprc"],
                               **{f"auroc_{c}": a for c, (a, _) in m["per_class"].items()}})
        for enc, name in product(Q_ENCODERS, METHOD_FNS):
            for s in Q_TEST_SPLITS:
                z = scaled(name, enc, s, sup)
                for variant, score in ((name, z), (f"{name}+prior", PRIOR[enc][s] + BETA[(enc, k, name)] * z)):
                    if variant.endswith("+prior") and RESIDUAL_PRIOR == "none": continue
                    m = macro_metrics(YS[s], score)
                    R_ROWS.append({"encoder": enc, "shot": k, "episode": ep, "head": variant, "split": s,
                                   "macro_auroc": m["macro_auroc"], "macro_auprc": m["macro_auprc"],
                                   **{f"auroc_{c}": a for c, (a, _) in m["per_class"].items()}})
        dev_now = pd.DataFrame(R_ROWS).query("shot == @k and episode == @ep and split == 'dev'")
        print(f"{k}-shot ep {ep + 1}/{Q_EPISODES} | DEV: " + ", ".join(f"{r.encoder[:4]}/{r.head}={r.macro_auroc:.3f}" for r in dev_now.itertuples())
              + f" | {elapsed()}")

# ----------------------------------------------------------------------------
# 6. Report (same format as the quantum cell): mean ± 95% t-CI, paired sign-flip tests, Holm
# ----------------------------------------------------------------------------
qdf = pd.DataFrame(R_ROWS); qdf.to_csv(RDIR / "per_episode_metrics.csv", index=False)

def ci95(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return float(student_t.ppf(.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else np.nan

def signflip_p(d, n_perm=PERMUTATIONS, seed=SEED):
    d = np.asarray(d, float); d = d[np.isfinite(d)]; obs = abs(d.mean())
    if len(d) == 0: return np.nan
    signs = np.array(list(product((1, -1), repeat=len(d)))) if len(d) <= 12 else np.random.default_rng(seed).choice((1, -1), size=(n_perm, len(d)))
    return float((np.abs((signs * d).mean(1)) >= obs - 1e-12).mean())

def holm(ps):
    order = np.argsort(ps); adj = np.empty(len(ps)); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

summ = qdf.groupby(["split", "shot", "encoder", "head"]).macro_auroc.agg(mean="mean", ci95=ci95, n="size").reset_index()
summ.to_csv(RDIR / "summary_retrieval.csv", index=False)
print("\n=== Q0 / Q1 / Q2 RETRIEVAL ABLATIONS (macro AUROC, mean ± 95% CI over episodes) ===\n" + summ.round(4).to_string(index=False))

def paired(split, k, a_enc, a, b_enc, b):
    g = qdf[(qdf["split"] == split) & (qdf["shot"] == k)]
    A = g[(g["encoder"] == a_enc) & (g["head"] == a)].set_index("episode").macro_auroc
    B = g[(g["encoder"] == b_enc) & (g["head"] == b)].set_index("episode").macro_auroc
    d = (A - B).dropna().to_numpy()
    return {"delta": d.mean() if len(d) else np.nan, "ci95": ci95(d), "p_perm": signflip_p(d), "wins": int((d > 0).sum()), "n": len(d)}

suf = "+prior" if RESIDUAL_PRIOR == "chexzero" else ""
M = {e: MATCHED_METHOD[e] + suf for e in MATCHED_METHOD}
matched = summ[summ.apply(lambda r: r["encoder"] in M and r["head"] == M[r["encoder"]], axis=1)]
print(f"\n=== HEADLINE: MATCHED metric (pre-adapted with m, scored with m) | prior={RESIDUAL_PRIOR} ===\n" + matched.round(4).to_string(index=False))
ref = summ[summ["head"] == "REF_chexzero_zeroshot"]
print("\nReference only (NOT part of Q0/Q1/Q2, NOT in any test): CheXzero zero-shot\n" + ref.round(4).to_string(index=False))
tests = []
for s, k in product(Q_TEST_SPLITS, Q_SHOTS):
    for e in ("pre_cosine", "pre_euclidean", "pre_mahalanobis"):          # retrieval-trained vs Q0 (no retrieval)
        tests.append({"family": "matched_vs_Q0", "split": s, "shot": k, "comparison": f"{e}/{M[e]} vs pre_none/{M['pre_none']}",
                      **paired(s, k, e, M[e], "pre_none", M["pre_none"])})
    for e in ("pre_euclidean", "pre_mahalanobis"):                        # metric-specific?
        tests.append({"family": "matched_metric_vs_Q1", "split": s, "shot": k, "comparison": f"{e}/{M[e]} vs pre_cosine/{M['pre_cosine']}",
                      **paired(s, k, e, M[e], "pre_cosine", M["pre_cosine"])})
    for e in ("pre_cosine", "pre_euclidean", "pre_mahalanobis"):          # does TRAINING with the metric help that metric?
        tests.append({"family": "metric_training_effect", "split": s, "shot": k, "comparison": f"{M[e]}: {e} vs pre_none",
                      **paired(s, k, e, M[e], "pre_none", M[e])})
    for e in MATCHED_METHOD:                                               # pre-adaptation vs NIH-only encoders
        tests.append({"family": "preadapt_vs_original", "split": s, "shot": k, "comparison": f"{M[e]}: {e} vs original",
                      **paired(s, k, e, M[e], "original", M[e])})
tdf = pd.DataFrame(tests); tdf["p_holm"] = np.nan
for (fam, s), g in tdf.groupby(["family", "split"]):
    tdf.loc[g.index, "p_holm"] = holm(g.p_perm.fillna(1.0).to_numpy())
tdf.to_csv(RDIR / "significance.csv", index=False)
print("\n=== PAIRED TESTS (episode-level sign-flip permutation, Holm within family x split) ===\n" + tdf.round(4).to_string(index=False))
print(f"\nDONE -> {RDIR} | {elapsed()} | ablation claims need supervisor review.")

if SMOKE:
    mins = (time.time() - RUN_T0) / 60
    init = qdf[qdf.split == "dev"].groupby(["encoder", "head"]).macro_auroc.mean().unstack("encoder").round(3)
    print("\n" + "=" * 78 + f"\nSMOKE INITIAL RESULT ({mins:.1f} min, budget {SMOKE_BUDGET_MIN} min)"
          + ("" if mins <= SMOKE_BUDGET_MIN else "  <-- over budget") + "\n" + "=" * 78)
    print("Frozen audit (DEV):\n" + AUDIT[AUDIT.split == "dev"].round(3).to_string(index=False))
    print(f"\n{Q_SHOTS[0]}-shot DEV macro AUROC by method (mean of {Q_EPISODES} episodes; prior={RESIDUAL_PRIOR}):\n" + init.to_string())
    print("\n~40-patient cohorts: these only show the pipeline runs end to end. Set RUN_MODE = 'FULL' in Cell 1 and Run All.")

beta (+prior variant, tuned on SELECT): {'original|1|Q0_stageA': 1.0, 'original|1|Q1_cosine': 1.0, 'original|1|Q2_euclidean': 1.0, 'original|1|Q2_mahalanobis': 1.0, 'original|5|Q0_stageA': 1.0, 'original|5|Q1_cosine': 1.0, 'original|5|Q2_euclidean': 1.0, 'original|5|Q2_mahalanobis': 1.0, 'pre_none|1|Q0_stageA': 1.0, 'pre_none|1|Q1_cosine': 1.0, 'pre_none|1|Q2_euclidean': 1.0, 'pre_none|1|Q2_mahalanobis': 1.0, 'pre_none|5|Q0_stageA': 1.0, 'pre_none|5|Q1_cosine': 1.0, 'pre_none|5|Q2_euclidean': 1.0, 'pre_none|5|Q2_mahalanobis': 1.0, 'pre_cosine|1|Q0_stageA': 1.0, 'pre_cosine|1|Q1_cosine': 1.0, 'pre_cosine|1|Q2_euclidean': 1.0, 'pre_cosine|1|Q2_mahalanobis': 1.0, 'pre_cosine|5|Q0_stageA': 1.0, 'pre_cosine|5|Q1_cosine': 1.0, 'pre_cosine|5|Q2_euclidean': 1.0, 'pre_cosine|5|Q2_mahalanobis': 1.0, 'pre_euclidean|1|Q0_stageA': 1.0, 'pre_euclidean|1|Q1_cosine': 1.0, 'pre_euclidean|1|Q2_euclidean': 1.0, 'pre_euclidean|1|Q2_mahalanobis': 1.0, 'pre_euclidean|5|Q0_stageA': 1.0, 'pre_euclidean|5|Q1_c